# Phase 7: Temporal Forecasting

This notebook analyzes the frozen Phase 4-6 Ridge/OLS predictions over time. It does not refit models, retune parameters, or alter the Phase 0-6 experiments.

## Frozen context and validation strategy

Validation temporal methodology is developed from ID16 and ID22 using persisted Phase 4 selected-model predictions. ID18 and ID19 are displayed only as post-hoc held-out observations from persisted Phase 6 predictions after the methodology was fixed. The trailing window is 12 samples x 5 seconds = 60 seconds.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
results_dir = project_root / 'results' / 'freetwinev'
figures_dir = results_dir / 'figures'
validation = pd.read_csv(results_dir / 'phase7_temporal_predictions_validation.csv')
test_posthoc = pd.read_csv(results_dir / 'phase7_temporal_predictions_test_posthoc.csv')
per_run = pd.read_csv(results_dir / 'phase7_temporal_metrics_per_run.csv')
rolling = pd.read_csv(results_dir / 'phase7_rolling_error_summary.csv')
regime = pd.read_csv(results_dir / 'phase7_regime_error_summary.csv')
audit = json.loads((results_dir / 'phase7_audit.json').read_text(encoding='utf-8'))
print('Audit:', audit['overall_status'])

## Temporal forecasting methodology

Point forecasts are the already-selected Extended 21-feature Ridge model at 60 seconds and OLS model at 300 seconds. Trajectories use time_s/time_min. Rolling MAE and RMSE use a trailing 12-observation window within each physical run. No rows cross run boundaries. Regime analysis uses Spearman correlations between absolute error and frozen Extended features; the descriptive high-load threshold is the validation 75th percentile of current_abs.

In [ ]:
validation[['split', 'exp_id', 'time_min', 'horizon', 'target_actual', 'target_predicted', 'residual', 'rolling_mae_60s', 'rolling_rmse_60s']].head()

## Forecast trajectories

The runner-generated figures show actual DeltaT, predicted DeltaT, and the evolution of error separately for both horizons and validation/test runs.

In [ ]:
names = ['36_phase7_validation_trajectories.png', '37_phase7_test_trajectories_posthoc.png']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, name in zip(axes, names):
    ax.imshow(plt.imread(figures_dir / name)); ax.set_title(name); ax.axis('off')
plt.tight_layout()

## Rolling temporal error

The 60-second trailing window makes local error trends comparable with the primary forecast horizon. Initial rows before 12 samples have no complete rolling statistic and are retained as missing rather than padded.

In [ ]:
rolling

## Horizon comparison and per-run behavior

The per-run table separates ID16/ID22 validation behavior from ID18/ID19 post-hoc test behavior. The 60-second and 300-second error distributions are not treated as interchangeable.

In [ ]:
per_run

## Operating-regime analysis

Validation correlations and the validation-derived high-load descriptive subset are the development evidence. Test regime rows are labeled post-hoc and were not used to modify the method.

In [ ]:
regime[regime['Split'] == 'validation']

In [ ]:
plt.figure(figsize=(10, 6)); plt.imshow(plt.imread(figures_dir / '39_phase7_regime_error_correlations.png')); plt.axis('off'); plt.show()

## Validation findings

Validation findings are based only on ID16 and ID22 persisted predictions. They quantify temporal error, rolling stability, per-run differences, and relationships with available frozen telemetry. No model or feature was changed in response.

In [ ]:
assert audit['overall_status'] == 'PASS'
assert audit['checks']['validation_decisions_precede_test_access']
assert audit['checks']['no_test_tuning']
assert audit['checks']['feature_contract']['feature_count'] == 21
audit['checks']

## Held-out test reporting

ID18 and ID19 results are post-hoc reporting only. They are not validation data and were not used for temporal-window choice, thresholds, model selection, calibration, or error-driven modification.

In [ ]:
test_posthoc[['exp_id', 'time_min', 'horizon', 'target_actual', 'target_predicted', 'residual']].head()

## Limitations and conclusion

The temporal analysis is diagnostic rather than a new forecasting model. Only two validation and two held-out runs are available. Rolling summaries depend on the fixed sampling interval and are not a substitute for independent experiments. Phase 7 leaves the selected models unchanged and is complete; the project may proceed to Phase 8 only after explicit authorization.